# [16.3] Shapley Interactions with shapiq

This notebook moves from single-feature Shapley credit to pairwise interaction
credit. You will implement exact second-order Shapley interaction indices on
complete coalition tables, then compare your result against `shapiq` SII.

## Core question

Can exact Shapley interaction indices recover planted feature synergies and
separate them from additive effects before we trust a library result?

## Learning objectives

By the end, you should be able to:

1. Build complete finite coalition tables for additive and interacting games.
2. Implement exact pairwise SII from the discrete second derivative formula.
3. Compare your exact implementation with `shapiq` on the same finite game.
4. Interpret CUDA ablation metrics with positive, negative, and shuffled controls.

> Difficulty: 🔴🔴🔴🔴⚪  
> Importance: 🔵🔵🔵🔵⚪

<img src="../../instructions/assets/shapley_interactions_validation_loop.svg" width="760">

The point is not to call an attribution library and trust the output. The point
is to build the finite-table interaction object first, use zero-interaction and
planted-pair controls, and only then treat `shapiq` as a parity check.

<details><summary>Help - what is this notebook trying to prove?</summary>

It proves local pairwise interaction recovery on complete finite games. The
final CUDA report extends that exact control to real ablations from a trained
finite neural model organism. It does not claim broad large-model interaction
attribution.

</details>


## Setup

Run the setup cell once. The tests are small because this section is about
exactness and interpretation, not about throughput.

<details><summary>Expected output</summary>

No printed output. Imports should succeed and the report dataclasses should be
defined.

</details>


In [ ]:
from collections.abc import Callable, Mapping
from dataclasses import dataclass
import itertools
import json
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part3_shapley_interactions_shapiq"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_shapley_interactions_shapiq.tests as tests

Coalition = frozenset[int]


@dataclass(frozen=True)
class PairwiseInteractionReport:
    pair_interactions: t.Tensor
    target_pair: tuple[int, int]
    target_interaction: float
    max_spurious_interaction: float
    recovers_interaction: bool


@dataclass(frozen=True)
class ShapiqInteractionParityReport:
    exact_pair_interactions: t.Tensor
    shapiq_pair_interactions: t.Tensor
    max_abs_error: float
    matches_shapiq: bool
    shapiq_available: bool


## Exercise 1 - build additive and interaction games

Implement complete coalition tables. Additive games should have no pairwise
interactions. A planted interaction game should add a bonus only when both
players in `pair` are present.

<details><summary>Help - why include the empty coalition?</summary>

The empty coalition is the baseline. Leaving it out changes the game and breaks
the weighting logic later.

</details>

<details><summary>Common bugs</summary>

- Returning only nonempty coalitions.
- Letting a pair contain the same player twice.
- Forgetting to validate the length of `additive_weights`.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_additive_game_enumerates_complete_zero_interaction_table` passed!
```

</details>

<details><summary>Solution</summary>

Enumerate every subset with `itertools.combinations`, then define additive and
pair-bonus value functions over those subsets.

</details>


In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    raise NotImplementedError()


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    raise NotImplementedError()


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    raise NotImplementedError()


def interaction_game(
    num_players: int,
    *,
    pair: tuple[int, int] = (0, 1),
    pair_weight: float = 1.0,
    additive_weights: t.Tensor | None = None,
) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_additive_game_enumerates_complete_zero_interaction_table(
    additive_game,
    pairwise_shapley_interactions=lambda values, num_players: t.zeros((num_players, num_players), dtype=t.float64),
)


## Exercise 2 - compute exact pairwise interactions

For each unordered pair `(i, j)`, average the second-order coalition delta over
background coalitions that contain neither player. Fill a symmetric matrix and
keep the diagonal zero.

<details><summary>Help - what is the second-order delta?</summary>

Measure how much extra value appears when `i` and `j` are added together beyond
adding each one separately: `v(Sij) - v(Si) - v(Sj) + v(S)`.

</details>

<details><summary>Common bugs</summary>

- Using ordinary Shapley weights instead of second-order SII weights.
- Including one of the pair players in the background coalition.
- Forgetting to mirror the matrix entry.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_additive_game_enumerates_complete_zero_interaction_table` passed!
All tests in `test_interaction_game_recovers_target_pair_delta` passed!
```

</details>

<details><summary>Solution</summary>

Use weight `k! * (n-k-2)! / (n-1)!` for background coalitions of size `k`, then
accumulate the second-order delta for each pair.

</details>


In [ ]:
def pairwise_shapley_interactions(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_additive_game_enumerates_complete_zero_interaction_table(
    additive_game,
    pairwise_shapley_interactions,
)
tests.test_interaction_game_recovers_target_pair_delta(
    interaction_game,
    pairwise_shapley_interactions,
)


## Exercise 3 - recover one target pair and reject off-target pairs

Build a report that checks the target pair value and the maximum absolute value
of every off-target pair.

<details><summary>Help - why check off-target pairs?</summary>

A method that recovers the intended pair but hallucinates extra interactions is
not an interaction detector. The negative-control pairs matter as much as the
positive pair.

</details>

<details><summary>Common bugs</summary>

- Checking only the target pair.
- Treating `(i, j)` and `(j, i)` as separate claims.
- Returning pass when the target sign is wrong.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_pairwise_interaction_report_matches_reference_and_rejects_spurious_pairs` passed!
```

</details>

<details><summary>Solution</summary>

Compute the full matrix, read the target entry, scan all other unordered pairs,
and require both target accuracy and low off-target mass.

</details>


In [ ]:
def pairwise_interaction_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    target_pair: tuple[int, int] = (0, 1),
    expected_target: float = 1.0,
    tolerance: float = 1e-9,
) -> PairwiseInteractionReport:
    raise NotImplementedError()


tests.test_pairwise_interaction_report_matches_reference_and_rejects_spurious_pairs(
    pairwise_interaction_report,
)


## Exercise 4 - compare exact interactions to shapiq SII

Use `shapiq` as an independent implementation check on the same complete table.
The point is parity, not outsourcing the lesson.

<details><summary>Help - how does shapiq call your game?</summary>

It passes boolean coalition arrays. Convert each row to a `frozenset` of present
player indices before indexing the complete value table.

</details>

<details><summary>Common bugs</summary>

- Comparing against the wrong interaction index.
- Returning zeros when `shapiq` is missing.
- Forgetting that `shapiq` may include tiny numerical noise on off-target pairs.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_shapiq_interaction_parity_report_matches_exact_sii` passed!
```

</details>

<details><summary>Solution</summary>

Wrap your finite table as a callable game, run `shapiq.AgnosticExplainer` with
`index="SII"` and `max_order=2`, then copy pair values into a symmetric matrix.

</details>


In [ ]:
def shapiq_pairwise_interactions(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    index: str = "SII",
) -> t.Tensor:
    raise NotImplementedError()


def shapiq_interaction_parity_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    index: str = "SII",
    tolerance: float = 1e-6,
) -> ShapiqInteractionParityReport:
    raise NotImplementedError()


tests.test_shapiq_interaction_parity_report_matches_exact_sii(
    shapiq_interaction_parity_report,
)


## Exercise 5 - inspect the planted neural-game target

Before reading the trained-model report, check the analytic four-feature game.
It should contain a positive `(0, 2)` interaction and a negative `(1, 3)`
interaction.

<details><summary>Expected output</summary>

```text
All tests in `test_neural_game_value_table_contains_planted_interactions` passed!
```

</details>

<details><summary>Interpreting the result</summary>

This is the ground truth for the final CUDA claim. The trained model only earns
credit if its ablation table recovers these planted interactions and leaves
unplanted pairs near zero.

</details>


In [ ]:
tests.test_neural_game_value_table_contains_planted_interactions()


## Exercise 6 - assemble the notebook contract and read the report

The smoke contract is fast and exact. The CUDA report is slower and checks the
same interaction logic on a trained finite model organism.

<details><summary>Help - why keep the CUDA path separate?</summary>

The notebook contract isolates the algorithm you implemented. The CUDA report
asks whether the same algorithm still recovers planted interactions from real
model ablations.

</details>

<details><summary>Common bugs</summary>

- Returning tensors directly instead of JSON-ready lists.
- Treating toy `shapiq` parity as trained-model evidence.
- Forgetting to check the shuffled-label negative control.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_matches_shapley_interaction_contract` passed!
```

</details>

<details><summary>Solution</summary>

Convert tensor fields to lists, include additive, target-pair, and `shapiq`
parity checks, then assert the committed CUDA report fields directly.

</details>


In [ ]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def additive_interaction_smoke_test() -> dict:
    raise NotImplementedError()


def target_pair_interaction_smoke_test() -> dict:
    raise NotImplementedError()


def shapiq_parity_smoke_test() -> dict:
    raise NotImplementedError()


def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


def load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] is True and report["tests_passed"] is True
    assert gpu["cuda_available"] is True and gpu["preflight_passed"] is True
    assert gpu["positive_interaction_pair"] == [0, 2]
    assert gpu["negative_interaction_pair"] == [1, 3]
    assert gpu["interaction_max_abs_error"] <= 1e-4
    assert gpu["max_spurious_interaction"] <= 1e-4
    assert gpu["shapiq_matches"] is True
    assert gpu["shuffled_control_rejected"] is True
    return report


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    _ = max_vram_gb
    return load_committed_gpu_report()["metrics"]["gpu_test"]


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_notebook_contract(run_smoke_test)
tests.test_committed_gpu_report_matches_shapley_interaction_contract()


## Signature Result

<img src="../../instructions/assets/shapley_interactions_signature_result.svg" width="760">

| Check | Expected | Observed |
| --- | ---: | ---: |
| Complete coalition table | 16 | 16 |
| Neural fit MSE | <= 1e-8 | 1.102e-12 |
| Positive pair `(0, 2)` | 2.2 | 2.2000004 |
| Negative pair `(1, 3)` | -1.5 | -1.5000007 |
| Max interaction error | <= 1e-4 | 4.52e-7 |
| Max off-target interaction | <= 1e-4 | 6.61e-7 |
| `shapiq` parity error | <= 1e-5 | 1.18e-8 |
| Shuffled-label error | >= 1.0 | 4.3833 |
| Peak VRAM | <= 1.0 GB | 0.063 GB |

<details><summary>Help - how should this result be interpreted?</summary>

The result says exact pairwise interactions recover planted positive and
negative pair effects from a complete finite table produced by a trained CUDA
model. It does not say sampled interaction attribution is reliable on large
models.

</details>


## Limitations

- This is a GT-0 complete-table model organism, not broad large-model attribution.
- Pairwise interactions do not capture every higher-order interaction.
- `shapiq` parity is scoped to SII on the same finite value table.
- The trained model has four binary features and two planted pair interactions.

## Bonus anomaly hunting

- Add a three-way interaction game and inspect where pairwise SII misses it.
- Reduce the `shapiq` budget below the complete coalition count and measure error.
- Compare pairwise Shapley interactions against full-minus-ablated patching effects.
